# TopoDiffusionNet en miniature — les briques

Sans modèle entraîné, sur CPU en quelques secondes :
1. les données Shapes et la vérification des comptages ;
2. l'homologie persistante sur un exemple jouet (format de `cripser.computePH`) ;
3. un test jouet de L_top : optimiser directement les pixels d'une image.

## 0. Installation
Sur Colab : installe les dépendances, monte Google Drive et se place dans le dossier du projet. En local, la cellule ne fait rien (on reste dans le dossier du notebook). Si `drive.mount` échoue dans VS Code, utiliser la commande « Colab: Mount Google Drive to Server » de la palette (`Ctrl+Shift+P`).

In [1]:
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q diffusers cripser
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/01 Études/Master MVA/Topologie/topodiffusion-mini"
    # si le code est dans un sous-dossier du même nom (comme sur le PC)
    if not os.path.exists(os.path.join(PROJECT_DIR, "data.py")):
        PROJECT_DIR = os.path.join(PROJECT_DIR, "topodiffusion-mini")
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)  # pour `from data import ...`
print("Colab" if IN_COLAB else "local", "|", os.getcwd())

local | c:\Mes Documents\Cours\MVA\Topologie\topodiffusion-mini\topodiffusion-mini


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from data import C_VALUES, count_components, make_shapes_image

## 1. Les données
Quelques masques par valeur de c, et vérification des comptages avec `scipy.ndimage.label` (masques générés à la volée, pas besoin du jeu de données).

In [ ]:
rng = np.random.default_rng(0)
n_cols = 6
fig, axes = plt.subplots(len(C_VALUES), n_cols, figsize=(1.6 * n_cols, 1.7 * len(C_VALUES)))
for row, c in enumerate(C_VALUES):
    for col in range(n_cols):
        mask = make_shapes_image(c, rng)
        ax = axes[row, col]
        ax.imshow(mask, cmap="gray")
        ax.set_title(f"c={c}, compté={count_components(mask)}", fontsize=7)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# vérification sur 200 masques par valeur de c
n_per_c = 200
n_ok = sum(count_components(make_shapes_image(c, rng)) == c for c in list(C_VALUES) * n_per_c)
print(f"{n_ok} / {n_per_c * len(C_VALUES)} masques avec exactement c composantes")

## 2. Homologie persistante sur un exemple jouet
Deux gaussiennes + bruit : diagramme de persistance, paires (maximum, col), vérification du format de `cripser.computePH`.

In [ ]:
import cripser
from topo_loss import persistence_pairs

yy, xx = np.mgrid[:64, :64]

def bump(i, j, height, width=50):
    return height * np.exp(-((yy - i) ** 2 + (xx - j) ** 2) / width)

img = bump(15, 40, 1.0) + bump(45, 20, 0.8) + 0.05 * np.random.default_rng(0).standard_normal((64, 64))

# format brut : sous-niveaux de -img, donc les maxima de img ont les naissances les plus négatives
ph = cripser.computePH(-img, maxdim=0)
np.set_printoptions(suppress=True, precision=3)
print("[dim, birth, death, i_b, j_b, k_b, i_d, j_d, k_d], 3 premières naissances :")
print(ph[np.argsort(ph[:, 1])[:3]])

births, deaths, pers = persistence_pairs(img)
print(f"\n{len(pers)} paires ; persistances les plus grandes : {pers[:5]}")

In [ ]:
fig, (ax_img, ax_diag) = plt.subplots(1, 2, figsize=(10, 4.5))

# scatter attend (x, y) = (colonne, ligne)
ax_img.imshow(img, cmap="viridis")
ax_img.scatter(births[:2, 1], births[:2, 0], c="red", marker="^", label="maxima m_p (2 plus persistants)")
ax_img.scatter(deaths[:2, 1], deaths[:2, 0], c="white", marker="x", label="cols s_p")
ax_img.legend(fontsize=7, loc="lower right")
ax_img.set_title("image jouet")

birth_vals = img[births[:, 0], births[:, 1]]
death_vals = img[deaths[:, 0], deaths[:, 1]]
ax_diag.scatter(birth_vals, death_vals, s=8)
lo, hi = img.min(), img.max()
ax_diag.plot([lo, hi], [lo, hi], "k--", lw=1)
ax_diag.set_xlabel("naissance (hauteur du maximum)")
ax_diag.set_ylabel("mort (hauteur du col)")
ax_diag.set_title("diagramme de persistance (sur-niveaux)")
plt.tight_layout()
plt.show()

## 3. Test jouet de L_top
Optimiser directement les pixels d'une image bruitée avec L_top : on doit voir apparaître exactement c taches.

In [ ]:
from scipy import ndimage
from topo_loss import topo_loss_single

c_target = 3
# bruit lissé puis ramené dans [-1, 1] : beaucoup de bosses de tailles variées
init = ndimage.gaussian_filter(np.random.default_rng(1).standard_normal((64, 64)), 3)
init = init / np.abs(init).max()

x = torch.tensor(init, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([x], lr=0.02)
history = []
for it in range(300):
    loss = topo_loss_single(x, c_target)
    opt.zero_grad()
    loss.backward()
    opt.step()
    with torch.no_grad():
        x.clamp_(-1, 1)  # comme les vraies images ; sans ça, -S1 fait exploser les valeurs
    history.append(loss.item())

final = x.detach().numpy()
_, _, pers_final = persistence_pairs(final)
print("persistances les plus grandes après optimisation :", np.round(pers_final[:6], 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(init, cmap="viridis", vmin=-1, vmax=1)
axes[0].set_title(f"départ : {count_components(init > 0)} composantes > 0")
axes[1].imshow(final, cmap="viridis", vmin=-1, vmax=1)
axes[1].set_title(f"après L_top (c={c_target}) : {count_components(final > 0)} composantes > 0")
for ax in axes[:2]:
    ax.axis("off")
axes[2].plot(history)
axes[2].set_xlabel("itération")
axes[2].set_title("L_top")
plt.tight_layout()
plt.show()